# Génération de population — pipeline multi-taille avec checkpoints

Ce notebook orchestre la création de populations synthétiques pour plusieurs tailles cibles,
avec checkpoints intermédiaires dans un dossier `Temp/` situé à côté du notebook.

## Paramètres

- `POPULATION_SIZES` : liste des tailles à générer (multiples de 100)
- `FORCE_REGENERATE` : force la régénération depuis eqasim (ignore le cache de l'API)
- `FORCE_STEP` : force la reprise à partir d'une étape — `raw`, `fixed`, `pt_enriched`, `zone_enriched` ou `scheduled`

Le notebook va **jusqu'au bout** : il exporte vers `data/population/` (le seul dossier lu
par GAMA et le serveur d'agents), pose les traits imputés depuis les microdonnées EMC², et
rend un verdict de complétude. Rien à lancer à la main après lui.

## Pipeline avec checkpoints

| Étape | Entrée | Sortie |
|---|---|---|
| 1 – Génération eqasim | API eqasim | `Temp/1_raw/` |
| 2 – Validation activités | `Temp/1_raw/` | `Temp/2_fixed/` |
| 3 – Enrichissement PT | `Temp/2_fixed/` | `Temp/3_pt_enriched/` |
| 3bis – Enrichissement zone (AAV2020 + densité) | `Temp/3_pt_enriched/` | `Temp/4_zone_enriched/` |
| 4 – Calcul itinéraires OSMnx | `Temp/4_zone_enriched/` | `Temp/5_scheduled/` |
| 6 – Warm-up OSMnx | `Temp/5_scheduled/` | cache SQLite des routes |
| **7 – Export final** | `Temp/5_scheduled/` | **`data/population/`** |
| **8 – Traits imputés EMC²** | `data/population/` | `data/population/` |
| **9 – Audit de complétude** | `data/population/` | verdict complète / incomplète |

À chaque étape, si le fichier de sortie existe déjà dans `Temp/`, l'étape est ignorée.
Pour forcer la reprise à une étape donnée (et toutes les suivantes), définir `FORCE_STEP`.

## Prérequis

| Service | Commande | Port |
|---|---|---|
| eqasim | `docker compose up eqasim` | 8003 |

Dépendances Python : `numpy`, `pandas`, `tqdm`, `osmnx`, `python-calamine`

### Données INSEE requises (étape 3bis)

Placer dans `data/insee/` :

| Fichier | Source | Rôle |
|---|---|---|
| `fichier_diffusion_2026.xlsx` | [insee.fr/fr/statistiques/5040028](https://www.insee.fr/fr/statistiques/5040028) | Grille densité 2025 — colonne `DENS_AAV` (requis) |
| `AAV2020_au_01-01-2026.xlsx` | [insee.fr/fr/statistiques/5040879](https://www.insee.fr/fr/statistiques/5040879) | Catégorie pôle/couronne par commune (optionnel) |

La colonne `DENS_AAV` du fichier diffusion contient déjà le croisement densité × AAV :
`1=Urbain dense` · `2=Urbain intermédiaire` · `3=Rural périurbain` · `4=Rural non périurbain`

**Étape 3ter — Sélection stratifiée (scellement AAMAS).** Optionnelle (`SELECT_N`). Entre
l'enrichissement de zone et le routage : sélectionne `SELECT_N` personas dans le vivier généré,
par allocation proportionnelle aux 12 cellules couronne × motorisation de l'enquête
(`scripts/AAMAS/seal_population.py select`), exclut les domiciles hors périmètre, et fait
porter la suite de la chaîne sur `toulouse_population_<SELECT_N>_<SELECT_TAG>.json`. Le
scellement final (`seal`) se fait hors notebook, après l'étape 9.


In [1]:
import os, certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [2]:
# ── Paramètres ────────────────────────────────────────────────────────────────
POPULATION_SIZES      = [1000]      # multiples de 100 ; peut atteindre des centaines de milliers
GENERATE_PERSONALITY  = False           # True → génère les Big Five (lent)
FORCE_REGENERATE      = True           # True → appelle eqasim avec force=True + reécrit Temp/raw/
FORCE_STEP            = None            # None | 'raw' | 'fixed' | 'pt_enriched' | 'zone_enriched' | 'scheduled'
                                        # Force la reprise à partir de cette étape (et toutes les suivantes)

# ── Cadre de tirage (ticket 026) ──────────────────────────────────────────────
# Le périmètre d'enquête EMC² est une LISTE DE COMMUNES, pas un rectangle : c'est la seule
# façon de dire « ni plus ni moins ». PERIMETER=True le sert, et DEPARTMENTS le restreint.
#
# Le périmètre d'étude est celui des 453 communes sur SIX départements (ticket 031, option A) :
# DEPARTMENTS = None les sert tous — à condition que la BD TOPO et la BAN des départements 32,
# 81, 82, 09 et 11 soient dans eqasim-toulouse/data/ (le service refuse sinon, code 3, avec la
# liste de ce qui manque). ['31'] = répétition sur la Haute-Garonne seule (346 communes) : la
# 3ᵉ couronne y plafonne à 10,6 % de la population contre 15,4 % dans l'enquête
# (docs/arch/perimetre-population.md, limite n°6). Ne pas sceller une v4 sur ce cadre.
PERIMETER   = True
DEPARTMENTS = ['31']                      # None = les six départements du périmètre

BBOX = None                               # ignoré quand PERIMETER=True ; sinon rectangle WGS84
# BBOX = [1.35, 43.55, 1.50, 43.65]      # exemple : centre de Toulouse

CLEAR_DOWNSTREAM_ON_REGENERATE = True    # True → supprime les checkpoints downstream quand l'étape 1 régénère un raw

EQASIM_URL = 'http://localhost:8003'

# ── Sélection stratifiée (scellement AAMAS) ──────────────────────────────────
# Le service eqasim tire 15 % de plus que demandé et RENOMME le fichier à la taille
# demandée : « toulouse_population_1000.json » en contient 1 021. Pour un effectif rond ET
# représentatif, on génère un VIVIER (POPULATION_SIZES = [5000]) et on en sélectionne
# SELECT_N personas par allocation proportionnelle aux 12 cellules couronne × motorisation
# de l'enquête (scripts/AAMAS/seal_population.py select) — AVANT les étapes de routage
# (4+5, 6), qui croissent avec N et ne tournent alors que sur les retenus. La suite de la
# chaîne (export, traits EMC², audit) porte sur toulouse_population_<SELECT_N>_<SELECT_TAG>.json.
# Vivier : 5 000 donne 99,2 % de chances de remplir les 12 cellules ; 2 700 en donne 63 %.
# SELECT_N = None → pas de sélection, la chaîne continue sur le vivier entier.
SELECT_N   = None                       # ex. 1000
SELECT_TAG = 'AAMAS'                    # suffixe du fichier sélectionné
POPULATION_TAG = ''                     # posé par l'étape 3ter ; ne pas éditer à la main

# ── Routage : parallélisme et réchauffage ────────────────────────────────────
# Chaque worker charge sa copie des graphes OSMnx (0,3 à 1,5 Go). À 12 workers la machine
# de développement swappe (mesuré le 2026-09-02 : 23 Go de swap, workers à 50 % de CPU) et
# l'étape 6 dure des heures ; à 6 elle tient en RAM. L'étape 6 ne touche pas à la population :
# elle pré-calcule le cache d'itinéraires pour accélérer les runs. SKIP_WARMUP = True la saute
# — le runtime calcule les itinéraires manquants à la demande — et la relancer plus tard se
# fait avec SKIP_WARMUP = False, FORCE_STEP = None (les autres étapes sont alors sautées).
# Graphe du polygone des 453 communes (ticket 031 § 1.4) : ≈ 1 Go par worker chargé (mesuré, cf.
# docs/traces/<date>_graphe_osmnx_perimetre_453) ; 6 workers tiennent dans 32 Go avec le reste
# de la machine, 12 swappent. Le graphe se construit une fois : `make osmnx-perimeter-graph`.
MAX_WORKERS = 6
SKIP_WARMUP = True                      # défaut : sauté — à relancer quand un run en a besoin (décision 2026-09-02)
SELECTION_RULE = 'aamas_seal_v4'        # règle de sélection attendue à l'étape 3ter (scripts/AAMAS/seal_population.py)


## Initialisation de l'environnement

Création de l'arborescence de dossiers temporaires (`Temp/`) et définition des chemins vers les données sources et de sortie. Chaque étape écrit dans un sous-dossier numéroté (`1_raw/`, `2_fixed/`, …) pour permettre la reprise depuis n'importe quel point sans tout recalculer.

La fonction `should_force(step)` détermine si une étape doit être rejouée en tenant compte de `FORCE_REGENERATE` et `FORCE_STEP`.

In [3]:
# ── Chemins & dossiers Temp ───────────────────────────────────────────────────
import json
import os
import sys
import time
import urllib.request
import urllib.error
from pathlib import Path

REPO_ROOT    = Path('../../../').resolve()
POP_DIR      = REPO_ROOT / 'data' / 'population'
NOTEBOOK_DIR = Path('.').resolve()
TEMP_DIR     = NOTEBOOK_DIR / 'Temp'

TEMP_RAW       = TEMP_DIR / '1_raw'
TEMP_FIXED     = TEMP_DIR / '2_fixed'
TEMP_PT        = TEMP_DIR / '3_pt_enriched'
TEMP_ZONE      = TEMP_DIR / '4_zone_enriched'
TEMP_SCHEDULED = TEMP_DIR / '5_scheduled'

for d in [POP_DIR, TEMP_RAW, TEMP_FIXED, TEMP_PT, TEMP_ZONE, TEMP_SCHEDULED]:
    d.mkdir(parents=True, exist_ok=True)

# ── Utilitaires ───────────────────────────────────────────────────────────────
STEPS = ['raw', 'fixed', 'pt_enriched', 'zone_enriched', 'scheduled']

def should_force(step_name: str) -> bool:
    if FORCE_REGENERATE and step_name == 'raw':
        return True
    if FORCE_STEP is None or FORCE_STEP not in STEPS:
        return False
    return STEPS.index(step_name) >= STEPS.index(FORCE_STEP)

def pop_filename(n: int) -> str:
    # Le suffixe (POPULATION_TAG) est posé par l'étape 3ter après sélection : la population
    # sélectionnée a son propre nom et n'écrase jamais toulouse_population_<n>.json.
    tag = f'_{POPULATION_TAG}' if POPULATION_TAG else ''
    return f'toulouse_population_{n}{tag}.json'

def save_json(data, path: Path) -> None:
    tmp = path.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.rename(tmp, path)

def load_json(path: Path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def check_temporal_order(data: list) -> tuple[int, int]:
    """Check start_time ordering (at most 1 decreasing gap allowed for midnight wrap).
    Returns (n_persons_with_error, n_total_errors).
    """
    n_persons, n_total = 0, 0
    for person in data:
        acts = person.get('identity', {}).get('activities', [])
        n = len(acts)
        if n == 0:
            continue
        if n == 1:
            # Journée IMMOBILE (ticket 029) : une seule activité, domicile 0 → 86 400 s. Elle
            # est valide par construction ; le test « durée nulle » ci-dessous la rejetait
            # (86 400 mod 86 400 = 0) et l'export refusait toute population avec immobiles.
            continue
        errors = []
        for i, act in enumerate(acts):
            s, e = act.get('start_time'), act.get('end_time')
            if s is not None and e is not None and (e - s) % 86400 == 0:
                errors.append(f"act[{i}] durée nulle")
        nb_ecarts = sum(
            1 for i in range(n)
            if acts[(i - 1) % n].get('start_time', 0) > acts[i].get('start_time', 0)
        )
        if nb_ecarts > 1:
            errors.append(f"{nb_ecarts} écarts décroissants sur start_time")
        if errors:
            n_persons += 1
            n_total += len(errors)
    return n_persons, n_total

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'POP_DIR   : {POP_DIR}')
print(f'TEMP_DIR  : {TEMP_DIR}')
print(f'Tailles   : {POPULATION_SIZES}')

REPO_ROOT : /Users/yvesb/Documents/Projects/llm-agents-gama
POP_DIR   : /Users/yvesb/Documents/Projects/llm-agents-gama/data/population
TEMP_DIR  : /Users/yvesb/Documents/Projects/llm-agents-gama/scripts/data/population/Temp
Tailles   : [1000]


## Chargement des dépendances scientifiques

Import des bibliothèques de calcul numérique (`numpy`, `pandas`) et configuration des constantes partagées entre les étapes :

| Constante | Valeur | Rôle |
|---|---|---|
| `GTFS_STOPS` | `data/gtfs/tisseo_gtfs/stops.txt` | Arrêts Tisséo pour l'enrichissement TC |
| `OSMNX_CACHE` | `data/cache/osmnx/` | Cache des graphes routiers (évite le re-téléchargement) |
| `MAX_WORKERS` | 12 | Parallélisme du calcul de routes |
| `MAX_PT_DIST_M` | 1 500 m | Rayon de rattachement à un arrêt TC |

In [4]:
# ── Imports scientifiques & constantes ───────────────────────────────────────
import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

GTFS_STOPS     = REPO_ROOT / 'data' / 'gtfs' / 'tisseo_gtfs' / 'stops.txt'
OSMNX_CACHE       = REPO_ROOT / 'data' / 'cache' / 'osmnx'      # graphs (pickle) — local notebook
OSMNX_ROUTE_CACHE = REPO_ROOT / 'data' / 'cache' / 'osmnx'             # SQLite routes — chemin hôte
SCRIPTS_POP    = REPO_ROOT / 'scripts' / 'data' / 'population'
LLMAGENTS_PATH = str(REPO_ROOT / 'llm-agents')

# Graphe de routage des étapes 4+5 : le polygone des 453 communes (ticket 031 § 1.4), construit par
# scripts/data/population/build_osmnx_perimeter_graph.py — clé distincte du disque de 30 km de la
# production (`Toulouse, France_30000` → ecb40f20a303). Sans ce graphe, 98 des 154 agents de
# 3ᵉ couronne de la v3 se routaient sur un même nœud et recevaient une vitesse de repli.
from build_osmnx_perimeter_graph import PERIMETER_CACHE_KEY, PERIMETER_GRAPH_LABEL
CACHE_KEY     = PERIMETER_CACHE_KEY
if not (OSMNX_CACHE / f'graphs_{CACHE_KEY}.pkl').exists():
    raise FileNotFoundError(
        f'Graphe OSMnx du périmètre absent : {OSMNX_CACHE}/graphs_{CACHE_KEY}.pkl '
        f'({PERIMETER_GRAPH_LABEL}). Construisez-le : make osmnx-perimeter-graph')
# MAX_WORKERS : paramètre de la première cellule (papermill peut le surcharger)
MAX_PT_DIST_M = 1500.0

if LLMAGENTS_PATH not in sys.path:
    sys.path.insert(0, LLMAGENTS_PATH)
if str(SCRIPTS_POP) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_POP))

from population_utils import TRIP_MODES

print(f'GTFS_STOPS:  {GTFS_STOPS}')
print(f'OSMNX_CACHE: {OSMNX_CACHE}')
print(f'OSMnx cache key: {CACHE_KEY}  ({PERIMETER_GRAPH_LABEL})')
print(f'OSMnx route SQLite: {OSMNX_ROUTE_CACHE}')

GTFS_STOPS:  /Users/yvesb/Documents/Projects/llm-agents-gama/data/gtfs/tisseo_gtfs/stops.txt
OSMNX_CACHE: /Users/yvesb/Documents/Projects/llm-agents-gama/data/cache/osmnx
OSMnx cache key: ecb40f20a303
OSMnx route SQLite: /Users/yvesb/Documents/Projects/llm-agents-gama/data/cache/osmnx


## Vérification du service eqasim

Avant tout traitement, on s'assure que le service de génération de population eqasim répond sur `EQASIM_URL` (port 8003). En cas d'échec après 3 tentatives, le pipeline s'arrête immédiatement avec un message d'erreur explicite.

> **Prérequis** : `docker compose up eqasim` doit être lancé avant d'exécuter cette cellule.

In [5]:
# ── Vérification santé du service eqasim ─────────────────────────────────────
def check_health(url: str, retries: int = 3, delay: float = 2.0) -> bool:
    for i in range(retries):
        try:
            with urllib.request.urlopen(f'{url}/health', timeout=5) as r:
                return r.status == 200
        except Exception as e:
            print(f'  tentative {i+1}/{retries} : {e}')
            if i < retries - 1:
                time.sleep(delay)
    return False

if check_health(EQASIM_URL):
    print(f'Service eqasim OK → {EQASIM_URL}')
else:
    raise RuntimeError(
        f'Service eqasim inaccessible sur {EQASIM_URL}. '
        'Démarrez le service avec : docker compose up eqasim'
    )

Service eqasim OK → http://localhost:8003


---
## Étape 1 — Génération eqasim → `Temp/1_raw/`

Appel à l'API eqasim pour chaque taille de population définie dans `POPULATION_SIZES`. eqasim synthétise des agents toulousains avec leurs activités quotidiennes (domicile, travail, loisirs…) à partir de données INSEE et d'enquêtes de déplacements (EMC²).

- **Cache** : si le fichier existe déjà dans `Temp/1_raw/` et que `FORCE_REGENERATE=False`, l'appel API est ignoré.
- **Sortie** : un fichier JSON par taille contenant la liste des personnes avec leurs activités brutes et leurs attributs socio-démographiques.
- **Validation** : après génération, l'ordre temporel des activités est vérifié — au plus un écart décroissant est toléré (passage minuit).

**Cadre de tirage (ticket 026).** Avec `PERIMETER = True`, eqasim ne tire plus dans un
rectangle ni dans un département entier mais dans la **liste des communes du périmètre
d'enquête EMC²** — 346 communes avec `DEPARTMENTS = ['31']`, 453 avec `DEPARTMENTS = None`.
Le `sampling_rate` est recalculé sur la population RP 2022 de ces communes.

⚠ Avec `DEPARTMENTS = ['31']`, la chaîne est une **répétition** sur la Haute-Garonne : les
107 communes des cinq autres départements du périmètre demandent leur BD TOPO et leur BAN
(ticket 031 § 1.0, accord de l'auteur du dépôt). La limite est chiffrée et publiée —
`docs/arch/perimetre-population.md`, limite n°6 : la 3ᵉ couronne plafonne à 10,6 % de la
population au lieu de 15,4 %. Le service eqasim journalise le cadre retenu par département et
refuse de générer si un département demandé n'a pas ses données.


In [6]:
# ── Étape 1 — Génération eqasim → Temp/1_raw/ ────────────────────────────────
print('=' * 60)
print('ÉTAPE 1 — Génération eqasim → Temp/1_raw/')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname    = pop_filename(pop_size)
    raw_path = TEMP_RAW / fname

    if not should_force('raw') and raw_path.exists():
        size_mb = raw_path.stat().st_size / 1_048_576
        print(f'[SKIP] {fname}  ({size_mb:.1f} Mo) — déjà dans Temp/1_raw/')
        continue

    payload = {
        'population_size':      pop_size,
        'generate_personality': GENERATE_PERSONALITY,
        'force':                FORCE_REGENERATE,
    }
    if PERIMETER:
        # Prime sur BBOX côté serveur : un rectangle ne peut pas exprimer le périmètre.
        payload['perimeter'] = True
        if DEPARTMENTS is not None:
            payload['departments'] = DEPARTMENTS
    elif BBOX is not None:
        payload['bbox'] = BBOX

    body = json.dumps(payload).encode()
    req  = urllib.request.Request(
        f'{EQASIM_URL}/generate',
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )

    print(f'[GEN]  {fname}  (population_size={pop_size})…')
    t0 = time.monotonic()

    try:
        with urllib.request.urlopen(req, timeout=7200) as resp:
            result = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        result = json.loads(e.read())
        raise RuntimeError(f'eqasim HTTP {e.code} : {result}')

    elapsed = time.monotonic() - t0
    if result.get('status') != 'ok':
        raise RuntimeError(f'Échec eqasim pour {pop_size} agents : {result}')

    eqasim_out = POP_DIR / fname
    if not eqasim_out.exists():
        raise FileNotFoundError(f'Fichier eqasim introuvable : {eqasim_out}')

    data = load_json(eqasim_out)
    save_json(data, raw_path)
    size_mb = raw_path.stat().st_size / 1_048_576
    print(f'       {len(data)} personnes en {elapsed:.1f}s  ({size_mb:.1f} Mo) → Temp/1_raw/{fname}')
    if CLEAR_DOWNSTREAM_ON_REGENERATE:
        # TEMP_ZONE compris : sans lui, l'étape 3bis « sautait » sur le checkpoint pré-imputé de
        # l'ancien vivier et la sélection repartait d'une population que l'étape 1 venait de
        # remplacer (constaté le 2026-09-03 sur le vivier v3 → v4).
        for _dl_dir in [TEMP_FIXED, TEMP_PT, TEMP_ZONE, TEMP_SCHEDULED]:
            _p = _dl_dir / fname
            if _p.exists():
                _p.unlink()
                print(f'       [CASCADE] Supprimé {_dl_dir.name}/{fname}')
        _sqlite = OSMNX_CACHE / Path(fname).stem / 'osmnx_cache.db'
        if _sqlite.exists():
            _sqlite.unlink()
            print(f'       [CASCADE] Supprimé cache SQLite OSMnx : {_sqlite.relative_to(REPO_ROOT)}')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 1 terminée.')

ÉTAPE 1 — Génération eqasim → Temp/1_raw/
[GEN]  toulouse_population_1000.json  (population_size=1000)…
       1021 personnes en 113.1s  (3.1 Mo) → Temp/1_raw/toulouse_population_1000.json
       [CASCADE] Supprimé 2_fixed/toulouse_population_1000.json
       [CASCADE] Supprimé 3_pt_enriched/toulouse_population_1000.json
       [CASCADE] Supprimé 5_scheduled/toulouse_population_1000.json
       [CASCADE] Supprimé cache SQLite OSMnx : data/cache/osmnx/toulouse_population_1000/osmnx_cache.db
       [OK]   Ordre temporel : 1021/1021 personnes valides

Étape 1 terminée.


---
## Étape 2 — Validation & correction des activités → `Temp/2_fixed/`

Les données brutes eqasim peuvent contenir des séquences d'activités invalides (chevauchements, durées nulles, enchaînements incohérents). Cette étape applique deux passes de nettoyage :

1. **Correction des violations de séquence** via `fix_activities` : réajustement des bornes temporelles pour éliminer chevauchements et durées négatives.
2. **Fusion des activités redondantes** : deux activités consécutives (ou circulaires) de même `purpose` et même localisation sont fusionnées en une seule, pour éviter des micro-trajets de durée nulle.

Chaque agent est re-validé après correction ; les cas encore invalides sont signalés mais n'interrompent pas le pipeline.

In [7]:
# ── Étape 2 — Validation & correction des activités → Temp/2_fixed/ ──────────
from population_utils import check_activities, fix_activities

print('=' * 60)
print('ÉTAPE 2 — Validation & correction des activités → Temp/2_fixed/')
print('=' * 60)

def _same_loc(a, b):
    la, lb = a.get('location'), b.get('location')
    if la is None or lb is None:
        return la is lb
    return la.get('lon') == lb.get('lon') and la.get('lat') == lb.get('lat')

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    raw_path   = TEMP_RAW   / fname
    fixed_path = TEMP_FIXED / fname

    if not should_force('fixed') and fixed_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/2_fixed/')
        continue

    if not raw_path.exists():
        raise FileNotFoundError(f'Fichier raw manquant — relancer étape 1 : {raw_path}')

    data = load_json(raw_path)

    # Correction des violations de séquence
    persons_fixed = 0
    fixed_data = []
    for person in data:
        fixed_person, _ = fix_activities(person)
        remaining = check_activities(fixed_person)
        if remaining:
            print(f'  !! person {person.get("person_id", "?")} : encore invalide après correction : {remaining}')
        fixed_data.append(fixed_person)
        persons_fixed += 1

    # Fusion : activités consécutives ET circulaires partageant même purpose ET même location
    n_merged = 0
    for person in fixed_data:
        acts = person.get('identity', {}).get('activities', [])

        # Fusion consécutive
        i = 0
        while i < len(acts) - 1:
            if acts[i].get('purpose') == acts[i + 1].get('purpose') and _same_loc(acts[i], acts[i + 1]):
                acts[i]['end_time'] = acts[i + 1]['end_time']
                acts[i]['scheduled_start_time'] = None
                acts.pop(i + 1)
                n_merged += 1
            else:
                i += 1

        # Fusion circulaire : première et dernière
        if len(acts) >= 2 and acts[0].get('purpose') == acts[-1].get('purpose') and _same_loc(acts[0], acts[-1]):
            last = acts.pop()
            acts[0]['start_time'] = last['start_time']
            acts[0]['scheduled_start_time'] = None
            n_merged += 1

    save_json(fixed_data, fixed_path)
    print(f'[OK]   {fname} — {persons_fixed} corrigé(s), {n_merged} fusion(s) → Temp/2_fixed/')
    n_p, n_e = check_temporal_order(fixed_data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(fixed_data) - n_p}/{len(fixed_data)} personnes valides')

print()
print('Étape 2 terminée.')

ÉTAPE 2 — Validation & correction des activités → Temp/2_fixed/
[OK]   toulouse_population_1000.json — 1021 corrigé(s), 0 fusion(s) → Temp/2_fixed/
       [OK]   Ordre temporel : 1021/1021 personnes valides

Étape 2 terminée.


---
## Étape 3 — Enrichissement transports en commun → `Temp/3_pt_enriched/`

Pour chaque localisation d'activité, on calcule si elle se trouve à moins de `MAX_PT_DIST_M` (1 500 m) d'un arrêt Tisséo. Ce flag `public_transport` est ensuite utilisé par l'agent LLM pour décider du mode de déplacement pertinent (marche, voiture, TC…).

- **Source** : fichier `stops.txt` du GTFS Tisséo — 5 661 arrêts chargés en mémoire.
- **Méthode** : recherche du plus proche arrêt par distance euclidienne approximative sur les coordonnées WGS84.
- Le nombre de localisations enrichies est affiché par fichier.

In [8]:
# ── Étape 3 — Enrichissement des flags public_transport → Temp/3_pt_enriched/ ─
from population_utils import enrich_public_transport

print('=' * 60)
print('ÉTAPE 3 — Enrichissement public_transport → Temp/3_pt_enriched/')
print('=' * 60)

stops_df  = pd.read_csv(GTFS_STOPS, usecols=['stop_lat', 'stop_lon'])
stop_lats = stops_df['stop_lat'].values
stop_lons = stops_df['stop_lon'].values
print(f'Chargement GTFS : {len(stops_df)} arrêts')
print()

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    fixed_path = TEMP_FIXED / fname
    pt_path    = TEMP_PT    / fname

    if not should_force('pt_enriched') and pt_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/3_pt_enriched/')
        continue

    if not fixed_path.exists():
        raise FileNotFoundError(f'Fichier fixed manquant — relancer étape 2 : {fixed_path}')

    data = load_json(fixed_path)
    n = enrich_public_transport(data, stop_lats, stop_lons, MAX_PT_DIST_M)
    save_json(data, pt_path)
    print(f'[OK]   {fname} — {n} localisation(s) enrichie(s) → Temp/3_pt_enriched/')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 3 terminée.')

ÉTAPE 3 — Enrichissement public_transport → Temp/3_pt_enriched/
Chargement GTFS : 5661 arrêts

[OK]   toulouse_population_1000.json — 0 localisation(s) enrichie(s) → Temp/3_pt_enriched/
       [OK]   Ordre temporel : 1021/1021 personnes valides

Étape 3 terminée.


---
## Étape 3bis — Enrichissement zone urbaine/périurbaine/rurale → `Temp/4_zone_enriched/`

Pour chaque localisation d'activité, on détermine le type de zone à partir de deux tables INSEE :

| Source | Rôle |
|---|---|
| **AAV 2020** (Zonage en Aires d'Attraction des Villes) | Classification pôle / couronne / hors influence |
| **Grille de densité communale** | Niveau de densité 1–4 (dense → très peu dense) |

**Pipeline** :
1. Collecte de toutes les coordonnées uniques de la population
2. Géocodage inverse batch via l'API BAN (`api-adresse.data.gouv.fr`) → code INSEE commune
3. Jointure avec AAV2020 + grille de densité
4. Ajout du champ `zone_type` dans chaque `location` d'activité

**Valeurs possibles de `zone_type`** :

| Valeur | Signification |
|---|---|
| `urbain_dense` | Pôle AAV + commune dense ou intermédiaire |
| `urbain` | Pôle AAV + commune peu dense |
| `periurbain_dense` | Couronne AAV + commune dense ou intermédiaire |
| `periurbain` | Couronne/multipolarisé + commune peu dense |
| `bourg` | Hors influence AAV + commune dense |
| `rural` | Hors influence AAV + commune peu dense |
| `inconnu` | Code commune non trouvé dans les tables INSEE |

> **Prérequis** : placer dans `data/insee/` les fichiers `AAV2020_au_01-01-2023.csv` et `grille_communale_densite_2023.csv` (voir section "Données INSEE requises" en introduction).

In [9]:
# ── Étape 3bis — Enrichissement zone (AAV2020 + densité) → Temp/4_zone_enriched/ ─
import csv
import io
import ssl
import time as _time

INSEE_DIR          = REPO_ROOT / 'data' / 'insee'
DENSITE_FILE       = INSEE_DIR / 'fichier_diffusion_2026.xlsx'
AAV_FILE           = INSEE_DIR / 'AAV2020_au_01-01-2026.xlsx'
GEOCODE_CACHE_FILE = TEMP_DIR  / 'geocode_cache.json'

_SSL_CTX = ssl.create_default_context()
_SSL_CTX.check_hostname = False
_SSL_CTX.verify_mode    = ssl.CERT_NONE

print('=' * 60)
print('ÉTAPE 3bis — Enrichissement zone → Temp/4_zone_enriched/')
print('=' * 60)

if should_force('zone_enriched') and GEOCODE_CACHE_FILE.exists():
    GEOCODE_CACHE_FILE.unlink()
    print('[RESET] Cache géocodage supprimé (FORCE_STEP)')

if not DENSITE_FILE.exists():
    raise FileNotFoundError(
        f'Fichier densité manquant : {DENSITE_FILE}\n'
        f'Télécharger depuis https://www.insee.fr/fr/statistiques/5040028\n'
        f'et placer dans {INSEE_DIR}/'
    )

# ── Chargement grille de densité (DENS7) ─────────────────────────────────────
densite = pd.read_excel(
    DENSITE_FILE, sheet_name='Maille communale', header=4,
    dtype={'CODGEO': str}, engine='calamine',
)
densite.columns = [c.strip() for c in densite.columns]
print(f'Grille densité : {len(densite)} communes')

# Libellés naturels pour DENS7 — utilisés dans la phrase finale
DENS7_LABEL = {
    1: 'grand centre urbain',
    2: 'centre urbain intermédiaire',
    3: 'ceinture urbaine',
    4: 'petite ville',
    5: 'bourg rural',
    6: 'habitat rural dispersé',
    7: 'habitat rural très dispersé',
}
densite['_detail'] = densite['DENS7'].map(DENS7_LABEL).fillna('zone inconnue')
_densite_lookup    = densite.set_index('CODGEO')['_detail'].to_dict()

print('Distribution DENS7 :')
print(densite['_detail'].value_counts().to_string())
print()

# ── Chargement AAV2020 — ville-centre ─────────────────────────────────────────
_attraction_lookup: dict[str, str] = {}
if AAV_FILE.exists():
    try:
        aav = pd.read_excel(
            AAV_FILE, sheet_name='Composition_communale',
            header=5, dtype={'CODGEO': str}, engine='calamine',
        )
        aav.columns = [c.strip() for c in aav.columns]
        aav['_city'] = aav['LIBAAV2020'].where(aav['CATEAAV2020'] != '30', other='')
        _attraction_lookup = aav.set_index('CODGEO')['_city'].to_dict()
        print(f'AAV2020 : {len(_attraction_lookup)} communes')
    except Exception as e:
        print(f'[WARN] AAV2020 non chargé ({e})')
else:
    print('[INFO] AAV2020 absent — ville-centre ignorée')
print()

def _build_zone_label(codgeo: str) -> str:
    """
    Exemples :
      31555 → 'quartier de grand centre urbain sur la commune de Toulouse'
      31561 → 'quartier de ceinture urbaine sur la commune de Tournefeuille (aire de Toulouse)'
      09001 → 'bourg rural hors aire d'attraction urbaine'
    """
    label = _densite_lookup.get(codgeo, 'zone inconnue')
    city  = _attraction_lookup.get(codgeo, '')
    if not city:
        return f'{label} hors aire d\'attraction urbaine'
    return f'quartier de {label} sur la commune de {city}'

# ── Géocodage inverse batch (API BAN) — multipart/form-data ──────────────────
def _make_multipart(csv_bytes: bytes) -> tuple[bytes, str]:
    boundary = f'----BoundaryZone{int(_time.time())}'
    body = (
        f'--{boundary}\r\n'
        f'Content-Disposition: form-data; name="data"; filename="coords.csv"\r\n'
        f'Content-Type: text/csv\r\n\r\n'
    ).encode() + csv_bytes + f'\r\n--{boundary}--\r\n'.encode()
    return body, f'multipart/form-data; boundary={boundary}'

_geocode_cache: dict[tuple, str] = {}
if GEOCODE_CACHE_FILE.exists():
    _raw = load_json(GEOCODE_CACHE_FILE)
    _geocode_cache = {(float(k.split(',')[0]), float(k.split(',')[1])): v
                      for k, v in _raw.items()}
    print(f'Cache géocodage : {len(_geocode_cache)} coords')

def _save_geocode_cache() -> None:
    save_json({f'{k[0]},{k[1]}': v for k, v in _geocode_cache.items()}, GEOCODE_CACHE_FILE)

def _batch_reverse_geocode(coords: list[tuple[float, float]]) -> dict[tuple, str]:
    BATCH   = 5000
    result  = dict(_geocode_cache)
    missing = [c for c in coords if c not in result]
    if not missing:
        return result
    print(f'  {len(missing)} coords à géocoder ({len(coords) - len(missing)} en cache)…')
    for start in range(0, len(missing), BATCH):
        chunk     = missing[start:start + BATCH]
        csv_bytes = ('longitude,latitude\n' + '\n'.join(f'{lon},{lat}' for lon, lat in chunk)).encode()
        body, ctype = _make_multipart(csv_bytes)
        req = urllib.request.Request(
            'https://api-adresse.data.gouv.fr/reverse/csv/',
            data=body, headers={'Content-Type': ctype}, method='POST',
        )
        try:
            with urllib.request.urlopen(req, timeout=120, context=_SSL_CTX) as resp:
                reader = csv.DictReader(io.TextIOWrapper(resp, encoding='utf-8'))
                for row in reader:
                    try:
                        key = (float(row['longitude']), float(row['latitude']))
                        result[key] = row.get('result_citycode', '')
                        _geocode_cache[key] = result[key]
                    except (KeyError, ValueError):
                        pass
        except Exception as e:
            print(f'  [WARN] BAN API erreur (chunk {start}) : {e}')
        _save_geocode_cache()
        print(f'  Géocodé {min(start + BATCH, len(missing))}/{len(missing)} coords…')
    n_ok = sum(1 for v in result.values() if v)
    print(f'  {n_ok}/{len(coords)} codes commune résolus')
    return result

_STALE_FIELDS = {'zone_city', 'zone_density', 'zone_type', 'zone_type_detail',
                 'commune_pop', 'attraction_city', 'aav_category'}

# ── Traitement par taille de population ───────────────────────────────────────
for pop_size in POPULATION_SIZES:
    fname     = pop_filename(pop_size)
    pt_path   = TEMP_PT   / fname
    zone_path = TEMP_ZONE / fname

    if not should_force('zone_enriched') and zone_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/4_zone_enriched/')
        continue

    if not pt_path.exists():
        raise FileNotFoundError(f'Fichier pt_enriched manquant — relancer étape 3 : {pt_path}')

    data = load_json(pt_path)

    unique_locs = list({
        (act['location']['lon'], act['location']['lat'])
        for person in data
        for act in person.get('identity', {}).get('activities', [])
        if act.get('location') and act['location'].get('lon') is not None
    })
    print(f'{fname} : {len(unique_locs)} locations uniques à géocoder…')

    loc_to_codgeo = _batch_reverse_geocode(unique_locs)
    loc_to_zone   = {loc: _build_zone_label(codgeo) for loc, codgeo in loc_to_codgeo.items()}

    n_enriched = 0
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            loc = act.get('location')
            if not loc or loc.get('lon') is None:
                continue
            for f in _STALE_FIELDS:
                loc.pop(f, None)
            loc['zone'] = loc_to_zone.get((loc['lon'], loc['lat']), 'zone inconnue')
            n_enriched += 1

    save_json(data, zone_path)

    zone_counts: dict[str, int] = {}
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            z = act.get('location', {}).get('zone', 'zone inconnue')
            zone_counts[z] = zone_counts.get(z, 0) + 1
    print(f'[OK]   {fname} — {n_enriched} activités enrichies → Temp/4_zone_enriched/')
    top5 = sorted(zone_counts.items(), key=lambda x: -x[1])[:5]
    print(f'       Top 5 zones : {top5}')

print()
print('Étape 3bis terminée.')

ÉTAPE 3bis — Enrichissement zone → Temp/4_zone_enriched/
Grille densité : 34875 communes
Distribution DENS7 :
_detail
habitat rural dispersé         18288
habitat rural très dispersé     7236
bourg rural                     5072
petite ville                    1996
ceinture urbaine                 947
grand centre urbain              699
centre urbain intermédiaire      637

AAV2020 : 34875 communes

Cache géocodage : 2649 coords
[SKIP] toulouse_population_1000.json — déjà dans Temp/4_zone_enriched/

Étape 3bis terminée.


---

## Étape 3ter — Sélection stratifiée (scellement AAMAS)

Ne tourne que si `SELECT_N` est renseigné. Prend le vivier de `Temp/4_zone_enriched/`, pose la
couronne de résidence sur chaque persona si elle manque encore (même chemin que
`make residence-zone`), exclut les domiciles hors des 453 communes et les moins de 5 ans, puis
retient `SELECT_N` personas par allocation proportionnelle à la cible jointe couronne ×
motorisation (base personne, `scripts/AAMAS/cible_jointe_couronne_motorisation.yaml`). Ordre
intra-cellule déterministe (`sha256`), journal dans `*_selection.json`.

Un **déficit** (cellule que le vivier ne remplit pas) est comblé dans la même couronne puis
dans le vivier entier, journalisé, et rend le code 1 : la chaîne continue, mais le contrôle
final le verra. Après cette étape, `POPULATION_SIZES` et `POPULATION_TAG` désignent le fichier
sélectionné : les étapes 4 à 9 ne tournent que sur lui.


In [ ]:
# ── Étape 3ter — Sélection stratifiée AAMAS → Temp/4_zone_enriched/ ──────────
import subprocess

print('=' * 60)
print('ÉTAPE 3ter — Sélection stratifiée (scellement AAMAS)')
print('=' * 60)

if not SELECT_N:
    print('SELECT_N = None — pas de sélection, la chaîne continue sur le vivier entier.')
else:
    # Interpréteur des post-traitements : la sélection importe `llm_module` (shapely,
    # pyproj, geopandas) et `scripts.*` — même choix qu'à l'étape 8.
    _venv = REPO_ROOT / 'llm-agents' / '.venv' / 'bin' / 'python'
    _select_python = str(_venv) if _venv.exists() else sys.executable
    _env = {**os.environ, 'PYTHONPATH': str(REPO_ROOT)}

    for pop_size in POPULATION_SIZES:
        pool = TEMP_ZONE / pop_filename(pop_size)
        if not pool.exists():
            raise FileNotFoundError(f'Vivier manquant — relancer étape 3bis : {pool}')
        out = TEMP_ZONE / f'toulouse_population_{SELECT_N}_{SELECT_TAG}.json'
        # Une sélection existante n'est réutilisée que si elle vient de CE vivier : le journal
        # porte le sha256 du vivier, on le compare. Sinon (autre vivier, autre règle), on refait —
        # une sélection périmée servie en silence a déjà fait exporter une population v2 sous un
        # nom v3 (2026-09-03).
        _journal = out.with_name(out.stem + '_selection.json')
        _same_pool = False
        if out.exists() and _journal.exists():
            _pool_sha = hashlib.sha256(pool.read_bytes()).hexdigest()
            try:
                _prev = json.loads(_journal.read_text(encoding='utf-8'))
                _same_pool = _prev.get('vivier', {}).get('sha256') == _pool_sha and _prev.get('version') == SELECTION_RULE
            except (OSError, ValueError):
                _same_pool = False
        if _same_pool and not should_force('zone_enriched'):
            print(f'[SKIP] {out.name} — déjà sélectionné dans ce vivier (même sha256, règle {SELECTION_RULE})')
            continue
        if out.exists():
            print(f'[REFAIT] {out.name} — sélection existante issue d\'un autre vivier ou d\'une autre règle')
            for _stale in (out, _journal, TEMP_SCHEDULED / out.name):
                if _stale.exists():
                    _stale.unlink()
        # 3ter-a — PRÉ-IMPUTATION DU VIVIER (ticket 029). La sélection v3 équilibre aussi le
        # logement, le permis et l'abonnement TC : ces traits doivent exister sur le vivier
        # AVANT la sélection, sinon la marge est vide et la descente l'ignore. Mêmes scripts que
        # l'étape 8 (qui les rejouera à l'identique sur les retenus : déterministes par hachage),
        # en place sur le checkpoint. Codes 3 et 4 = pas des échecs (cf. étape 8).
        print(f'[PRÉ-IMPUTATION] {pool.name} : traits d\'équipement, logement et vélo sur le vivier…')
        for _label, _module in [('fix_minor_traits', 'scripts.data.population.fix_minor_traits'),
                                ('enrich_housing_type', 'scripts.data.population.enrich_housing_type'),
                                ('enrich_personal_bike', 'scripts.data.population.enrich_personal_bike'),
                                ('enrich_equipment', 'scripts.data.population.enrich_equipment'),
                                ('fix_minor_traits (2e passe)', 'scripts.data.population.fix_minor_traits')]:
            _t = time.monotonic()
            _proc = subprocess.run([_select_python, '-m', _module, str(pool)], cwd=str(REPO_ROOT), env=_env,
                                   capture_output=True, text=True)
            _rc = _proc.returncode
            print(f'         {_label:28s} code {_rc} ({time.monotonic() - _t:.0f}s)'
                  + ('' if _rc in (0, 3, 4) else ' ← ÉCHEC : ' + (_proc.stderr or '').strip().split('\n')[-1][:160]))
            if _rc not in (0, 3, 4):
                raise RuntimeError(f'Pré-imputation échouée : {_label} (code {_rc})')
        print(f'[SELECT] {SELECT_N} personas dans {pool.name} ({pop_size} demandés à eqasim)…')
        t0 = time.monotonic()
        proc = subprocess.run(
            [_select_python, '-m', 'scripts.AAMAS.seal_population', 'select',
             '--pool', str(pool), '--n', str(SELECT_N), '--out', str(out)],
            cwd=str(REPO_ROOT), env=_env, capture_output=True, text=True,
        )
        print(proc.stdout.rstrip() or '(pas de sortie)')
        if proc.returncode == 1:
            print('[DÉFICIT] vivier trop petit pour au moins une cellule — reports journalisés '
                  f'dans {out.stem}_selection.json ; la chaîne continue, le contrôle final le verra.')
        elif proc.returncode != 0:
            tail = (proc.stderr or '').strip().split('\n')[-8:]
            raise RuntimeError('Sélection échouée (code %d) :\n%s' % (proc.returncode, '\n'.join(tail)))
        print(f'         {time.monotonic() - t0:.1f}s → Temp/4_zone_enriched/{out.name}')

    # La suite de la chaîne porte sur le fichier sélectionné, et sur lui seul.
    POPULATION_SIZES = [SELECT_N]
    POPULATION_TAG = SELECT_TAG
    print()
    print(f'Étapes 4 à 9 : {pop_filename(SELECT_N)}')

print()
print('Étape 3ter terminée.')


---
## Étape 4 — Calcul des temps de trajet (scheduling) + Ajustement des horaires → `Temp/5_scheduled/`

Pour chaque personne, on calcule **un seul temps de trajet** par paire d'activités :
- `car` pour les propriétaires de voiture
- `bicycle` pour les autres

Ce temps est passé directement à `ajuster_planning` (paramètre `travel_times`) — il n'est **pas stocké dans le JSON**.
Le routage se fait sur les **graphes du polygone des 453 communes** (`make osmnx-perimeter-graph`,
clé `PERIMETER_CACHE_KEY`), pas sur le disque de 30 km de la production : un domicile de 3ᵉ couronne
y a des nœuds de graphe à lui (ticket 031 § 1.4).
Le cache SQLite OSMnx (foot / bicycle / car × 24h) est alimenté séparément lors du warm-up serveur.

- **Déduplication globale** : les paires communes à plusieurs tailles ne sont calculées qu'une seule fois.
- **Résultat** : plannings cohérents écrits directement dans `Temp/5_scheduled/`.


In [10]:
# ── Étape 4+5 — Temps de trajet scheduling + Ajustement des horaires ─────────
import multiprocessing

from population_utils import (
    collect_scheduling_pairs, build_travel_times, ajuster_planning,
)
from route_worker import init_worker, compute_route_worker

print('=' * 60)
print('ÉTAPES 4+5 — Scheduling routes + Ajustement des horaires')
print('=' * 60)

to_schedule = []
for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not should_force('scheduled') and scheduled_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/5_scheduled/')
        continue

    zone_path = TEMP_ZONE / fname
    if not zone_path.exists():
        raise FileNotFoundError(f'Fichier zone_enriched manquant — relancer étape 3bis : {zone_path}')

    to_schedule.append(pop_size)

if not to_schedule:
    print()
    print('Tous les fichiers sont déjà planifiés — étape ignorée.')
else:
    all_data: dict[int, list] = {}
    for pop_size in to_schedule:
        all_data[pop_size] = load_json(TEMP_ZONE / pop_filename(pop_size))

    # ── Collecte des paires de scheduling (1 mode par personne) ──────────────
    print()
    print('Collecte des paires de scheduling…')
    global_pairs: set = set()
    per_size_pairs: dict[int, set] = {}
    for pop_size, data in all_data.items():
        pairs = collect_scheduling_pairs(data)
        per_size_pairs[pop_size] = pairs
        global_pairs |= pairs
        print(f'  {pop_filename(pop_size)} : {len(pairs)} paires (scheduling mode)')

    print(f'Total unique global : {len(global_pairs)} paires à calculer')
    print()

    route_cache: dict[tuple, dict | None] = {}

    if global_pairs:
        # bicycle/foot : hour=None → passer 8h (résultat identique quelle que soit l'heure)
        tasks = [
            (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
            for lat1, lon1, lat2, lon2, mode, hour in global_pairs
        ]

        print(f'Calcul de {len(tasks)} routes avec {MAX_WORKERS} workers…')
        t0 = time.monotonic()

        ctx = multiprocessing.get_context('spawn')
        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            mp_context=ctx,
            initializer=init_worker,
            initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
        ) as pool:
            results = list(tqdm(
                pool.map(compute_route_worker, tasks, chunksize=20),
                total=len(tasks),
                desc='scheduling routes',
            ))

        elapsed = time.monotonic() - t0
        print(f'Terminé en {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

        ok_count = null_count = 0
        for args, result in results:
            lat1, lon1, lat2, lon2, mode, actual_hour = args
            route_cache[(lat1, lon1, lat2, lon2, mode, actual_hour if mode == 'car' else None)] = result
            if result is None:
                null_count += 1
            else:
                ok_count += 1
        print(f'Routes : {ok_count} OK, {null_count} inaccessibles (None)')
        print()

    # ── Build travel_times + ajuster_planning + sauvegarde ───────────────────
    for pop_size in to_schedule:
        fname = pop_filename(pop_size)
        data  = all_data[pop_size]

        travel_times_list = build_travel_times(data, route_cache)

        sched_errors = 0
        for entry, tt in zip(data, travel_times_list):
            acts = entry.get('identity', {}).get('activities', [])
            try:
                entry['identity']['activities'] = ajuster_planning(
                    fname, entry.get('person_id', '?'), acts,
                    travel_times=tt, raise_error=True,
                )
            except ValueError as exc:
                sched_errors += 1

        save_json(data, TEMP_SCHEDULED / fname)
        n_p, n_e = check_temporal_order(data)
        status = '[OK]  ' if n_e == 0 else '[WARN]'
        print(f'[OK]   {fname} — {len(data)} personnes → Temp/5_scheduled/')
        print(f'       {status} Ordre temporel : {len(data) - n_p}/{len(data)} valides')
        if sched_errors:
            print(f'       [WARN] {sched_errors} conflits de planning non résolus')

print()
print('Étapes 4+5 terminées.')

ÉTAPES 4+5 — Scheduling routes + Ajustement des horaires

Collecte des paires de scheduling…
  toulouse_population_1000 : 3899 paires (scheduling mode)
Total unique global : 3899 paires à calculer

Calcul de 3899 routes avec 12 workers…


scheduling routes:   0%|          | 0/3899 [00:00<?, ?it/s]

[worker pid=49550] graphs loaded in 128.4s  (simulation_date=2024-01-08)
[worker pid=49549] graphs loaded in 131.5s  (simulation_date=2024-01-08)
[worker pid=49547] graphs loaded in 137.2s  (simulation_date=2024-01-08)
[worker pid=49542] graphs loaded in 140.4s  (simulation_date=2024-01-08)
[worker pid=49551] graphs loaded in 140.9s  (simulation_date=2024-01-08)
[worker pid=49540] graphs loaded in 142.2s  (simulation_date=2024-01-08)
[worker pid=49546] graphs loaded in 143.2s  (simulation_date=2024-01-08)
[worker pid=49543] graphs loaded in 143.7s  (simulation_date=2024-01-08)
[worker pid=49545] graphs loaded in 143.7s  (simulation_date=2024-01-08)
[worker pid=49539] graphs loaded in 143.9s  (simulation_date=2024-01-08)
[worker pid=49541] graphs loaded in 144.0s  (simulation_date=2024-01-08)
[worker pid=49544] graphs loaded in 144.4s  (simulation_date=2024-01-08)


2026-08-24 17:39:27.110 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:27.145 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:30.538 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:33.007 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:33.657 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:36.123 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:40.623 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:40.818 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:44.237 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:39:44.316 | IN

Terminé en 737.4s  (189.1 ms/route)
Routes : 3856 OK, 43 inaccessibles (None)

[OK]   toulouse_population_1000.json — 1021 personnes → Temp/5_scheduled/
       [OK]   Ordre temporel : 1021/1021 valides

Étapes 4+5 terminées.


---
## Étape 5 — *(fusionnée avec l'étape 4)*

L'ajustement des horaires est désormais effectué directement dans l'étape 4.
Cette cellule est conservée pour la compatibilité du pipeline mais n'exécute rien.


In [11]:
# Étape 5 fusionnée avec étape 4 — rien à faire ici.
print("Étape 5 — déjà effectuée dans l'étape 4.")


Étape 5 — déjà effectuée dans l'étape 4.


---
## Étape 6 — Warm-up SQLite OSMnx → `data/osmnx_cache/toulouse_population_N/`

Pour chaque paire O-D unique de la population, on calcule **toutes les routes** nécessaires au serveur :
- `foot` × 1 entrée (indépendant du temps)
- `bicycle` × 1 entrée (indépendant du temps)
- `car` × 24 entrées horaires (congestion TomTom par plage)

Les résultats sont écrits dans le cache SQLite persistant (`osmnx_cache.db`).
Au démarrage du serveur, `get_direct_plan()` trouvera **100 % de hits** sur ce cache.

La cellule est **idempotente** : les routes déjà présentes en base sont ignorées.


In [12]:
# ── Étape 6 — Warm-up SQLite OSMnx ──────────────────────────────────────────
import multiprocessing
from datetime import datetime as _dt, date as _date

from population_utils import collect_warmup_pairs
from route_worker import init_worker, compute_route_worker
from trip_helper.osmnx_persistent_cache import OsmnxPersistentCache

_SIM_DATE = _date(2024, 1, 8)  # lundi — doit correspondre au jour de simulation

print('=' * 60)
print('ÉTAPE 6 — Warm-up SQLite OSMnx (foot + bicycle + car × 24h)')
print('=' * 60)

if SKIP_WARMUP:
    print('[SKIP] SKIP_WARMUP = True — réchauffage OSMnx sauté ; le runtime calculera les itinéraires')
    print('       manquants à la demande. Relancer avec SKIP_WARMUP = False (et MAX_WORKERS adapté à la RAM).')
    POPULATION_SIZES_WARMUP = []
else:
    POPULATION_SIZES_WARMUP = list(POPULATION_SIZES)

for pop_size in POPULATION_SIZES_WARMUP:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not scheduled_path.exists():
        print(f'[SKIP] {fname} — fichier scheduled manquant, relancer étape 4+5')
        continue

    data      = load_json(scheduled_path)
    all_pairs = collect_warmup_pairs(data)  # foot/bicycle × 1 + car × 24h

    _cache_dir = OSMNX_ROUTE_CACHE / Path(fname).stem
    _sqlite    = OsmnxPersistentCache(str(_cache_dir))

    # ── Vérification de couverture (idempotence) ───────────────────────────────
    pair_list = list(all_pairs)
    missing   = []
    for lat1, lon1, lat2, lon2, mode, hour in pair_list:
        actual_h  = hour if hour is not None else 8
        cdt       = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, *_   = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        if not _sqlite.lookup(key).found:
            missing.append((lat1, lon1, lat2, lon2, mode, hour))

    print(f'\n{fname}: {len(all_pairs)} paires total — {len(missing)} à calculer')

    if not missing:
        print(f'  [OK] Cache SQLite déjà complet.')
        continue

    # ── Calcul des routes manquantes ──────────────────────────────────────────
    tasks = [
        (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
        for lat1, lon1, lat2, lon2, mode, hour in missing
    ]

    print(f'  Calcul de {len(tasks)} routes avec {MAX_WORKERS} workers…')
    t0 = time.monotonic()

    ctx = multiprocessing.get_context('spawn')
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=ctx,
        initializer=init_worker,
        initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
    ) as pool:
        results = list(tqdm(
            pool.map(compute_route_worker, tasks, chunksize=20),
            total=len(tasks),
            desc=f'warmup {pop_size}',
        ))

    elapsed = time.monotonic() - t0
    print(f'  Terminé en {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

    # ── Écriture en SQLite ─────────────────────────────────────────────────────
    n_ok = n_null = 0
    for (lat1, lon1, lat2, lon2, mode, hour), (_, result) in zip(missing, results):
        actual_h              = hour if hour is not None else 8
        cdt                   = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, date_s, dow, bkt = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        _sqlite.store(key, date_s, dow, bkt, mode, lat1, lon1, lat2, lon2, result)
        if result is None:
            n_null += 1
        else:
            n_ok += 1

    print(f'  SQLite : {n_ok} OK, {n_null} inaccessibles → {_cache_dir.relative_to(REPO_ROOT)}/osmnx_cache.db')

print()
print('Étape 6 terminée.')


ÉTAPE 6 — Warm-up SQLite OSMnx (foot + bicycle + car × 24h)

toulouse_population_1000.json: 83478 paires total — 83478 à calculer
  Calcul de 83478 routes avec 12 workers…


warmup 1000:   0%|          | 0/83478 [00:00<?, ?it/s]

[worker pid=53001] graphs loaded in 94.8s  (simulation_date=2024-01-08)
[worker pid=53008] graphs loaded in 95.8s  (simulation_date=2024-01-08)
[worker pid=53000] graphs loaded in 96.7s  (simulation_date=2024-01-08)
[worker pid=53004] graphs loaded in 97.0s  (simulation_date=2024-01-08)
[worker pid=53002] graphs loaded in 97.1s  (simulation_date=2024-01-08)
[worker pid=52999] graphs loaded in 97.1s  (simulation_date=2024-01-08)
[worker pid=53003] graphs loaded in 97.1s  (simulation_date=2024-01-08)
[worker pid=53009] graphs loaded in 97.2s  (simulation_date=2024-01-08)
[worker pid=53005] graphs loaded in 97.4s  (simulation_date=2024-01-08)
[worker pid=53006] graphs loaded in 97.4s  (simulation_date=2024-01-08)
[worker pid=53007] graphs loaded in 97.6s  (simulation_date=2024-01-08)
[worker pid=52998] graphs loaded in 97.7s  (simulation_date=2024-01-08)


2026-08-24 17:51:21.011 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.011 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.013 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.015 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.015 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.015 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.014 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.014 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.016 | INFO     | trip_helper.osmnx_direct:<module>:64 - OSMnx HTTP concurrency limit: 3
2026-08-24 17:51:21.019 | IN

  Terminé en 9201.6s  (110.2 ms/route)
  SQLite : 82573 OK, 905 inaccessibles → data/cache/osmnx/toulouse_population_1000/osmnx_cache.db

Étape 6 terminée.


---
## Export final → `data/eqasim_output/`

Copie des fichiers planifiés depuis `Temp/5_scheduled/` vers le dossier de sortie définitif `data/eqasim_output/`. Un bilan de qualité est affiché pour chaque taille avant la copie :

| Indicateur | Signification |
|---|---|
| `missing_routes` | Agents dont au moins un trajet n'a pas pu être calculé (zones inaccessibles) |
| `missing_any_mode` | Agents dont le mode de transport reste indéterminé |
| `missing_pt` | Agents rattachés aux TC mais sans arrêt Tisséo dans le rayon de 1 500 m |

Les fichiers exportés sont directement consommables par la simulation GAMA et le serveur d'agents LLM.

In [13]:
# check_enrichment supprimé — les routes ne sont plus dans le JSON
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    path = TEMP_SCHEDULED / fname
    if not path.exists():
        print(f'[MISSING] {fname}')
        continue
    data = load_json(path)
    n_p, n_e = check_temporal_order(data)
    print(f'{fname}: {len(data)} personnes, {len(data)-n_p}/{len(data)} plannings valides')


toulouse_population_1000.json: 1021 personnes, 1021/1021 plannings valides


---
## Étape 7 — Export final `Temp/5_scheduled/` → `data/population/`

**C'est l'étape qui manquait au notebook**, et son absence était silencieuse : les cinq
étapes précédentes travaillaient dans `Temp/`, mais rien ne recopiait le résultat vers
`data/population/`, seul dossier lu par GAMA et par le serveur d'agents (monté dans les
conteneurs sous `/eqasim-output`). Le fichier qui y traînait était donc la **sortie brute
d'eqasim** déposée par l'étape 1 — sans correction d'activités, sans flag TC, et surtout
**sans horaires recalés** : `toulouse_population_1000.json` portait 0 activité planifiée
là où `Temp/5_scheduled/` en compte 3 944.

L'export vérifie avant de copier, et **refuse** plutôt que d'écraser une population
valide par une population dégradée : moins d'agents que la source brute, aucune activité
planifiée, ou ordre temporel invalide font échouer l'étape.

Une sauvegarde `.bak` de la version précédente est conservée à côté du fichier.


In [14]:
# ── Étape 7 — Export final → data/population/ ────────────────────────────────
import shutil

print('=' * 60)
print('ÉTAPE 7 — Export final → data/population/')
print('=' * 60)

EXPORTED = []
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    src   = TEMP_SCHEDULED / fname
    dst   = POP_DIR / fname

    if not src.exists():
        print(f'[MANQUE] {fname} absent de Temp/5_scheduled/ — étape 4+5 non jouée ?')
        continue

    data = load_json(src)
    n_agents     = len(data)
    n_activities = sum(len(p['identity']['activities']) for p in data)
    n_scheduled  = sum(1 for p in data for a in p['identity']['activities']
                       if a.get('scheduled_start_time') is not None)
    n_pt         = sum(1 for p in data for a in p['identity']['activities']
                       if (a.get('location') or {}).get('public_transport'))
    n_bad, _     = check_temporal_order(data)

    # Garde-fous : on n'écrase une population que par une population meilleure.
    problems = []
    if n_scheduled == 0:
        problems.append('aucune activité planifiée (étape 5 non jouée)')
    if n_bad:
        problems.append(f'{n_bad} planning(s) temporellement invalides')
    raw = TEMP_RAW / fname
    if raw.exists():
        n_raw = len(load_json(raw))
        if n_agents < n_raw * 0.5:
            problems.append(f'{n_agents} agents contre {n_raw} au brut — perte massive')
    if problems:
        print(f'[REFUS]  {fname} : ' + ' ; '.join(problems))
        print(f'         {dst.name} laissé en place, rien n\'a été écrasé.')
        continue

    if dst.exists():
        backup = dst.with_suffix('.json.bak')
        shutil.copy2(dst, backup)

    # Copie d'octets : on préserve exactement ce que l'étape 5 a écrit.
    shutil.copy2(src, dst)
    EXPORTED.append(fname)
    print(f'[OK]     {fname}  {n_agents} agents, {n_activities} activités, '
          f'{n_scheduled} planifiées, {n_pt} desservies en TC')
    print(f'         → {dst.relative_to(REPO_ROOT)}')

print()
print(f'Étape 7 terminée — {len(EXPORTED)}/{len(POPULATION_SIZES)} fichier(s) exporté(s).')


ÉTAPE 7 — Export final → data/population/
[OK]     toulouse_population_1000.json  1021 agents, 3944 activités, 3944 planifiées, 3218 desservies en TC
         → data/population/toulouse_population_1000.json

Étape 7 terminée — 1/1 fichier(s) exporté(s).


---
## Étape 8 — Traits imputés depuis les microdonnées EMC²

Quatre traits du persona ne viennent **ni** d'eqasim **ni** des tables INSEE : ils sont
lus ou imputés depuis l'enquête EMC² Toulouse 2023 (ProGEDO `lil-1750`), ou corrigés après
coup.
Sans cette étape, la population est syntaxiquement valide et **statistiquement fausse** —
c'est précisément le cas qui a produit un gradient d'équipement vélo inversé pendant des
mois.

| Script | Trait | Ce qu'il corrige |
|---|---|---|
| `fix_minor_traits` | `has_driving_license`, motifs, `car_availability` | permis d'enfants, motifs incohérents, voitures « à partager » à cause d'un permis de mineur |
| `enrich_housing_type` | `housing_type` | trait **absent** d'eqasim ; imputé par (zone fine, taille du ménage) |
| `enrich_personal_bike` | `personal_bike` | nombre de vélos **recopié** d'un ménage ENTD 2008 apparié sans la taille du foyer → gradient inversé |
| `enrich_residence_zone` | `residence_zone`, `residence_commune` | couronne **devinée à la distance** au Capitole au lieu d'être lue dans le découpage communal de l'enquête → 24,4 % des personas comparés à la cible d'une autre zone (ticket 021) |

**L'ordre compte** : `enrich_personal_bike` lit `housing_type` pour son rapport de
validation (croisement équipement × type d'habitat), il passe donc après
`enrich_housing_type`. `enrich_residence_zone` ne dépend de rien et passe en tête. Les
quatre sont **idempotents** — les rejouer ne coûte rien et ne change rien.

**Un trait à part** : `residence_zone` est **observé**, pas imputé — un domicile est dans
une commune ou il n'y est pas. Ni tirage, ni loi, ni sel : sa validation ne porte donc pas
sur une distribution mais sur un **accord** entre le classement par code de zone fine et le
classement par appartenance géométrique. Son code de sortie `4` n'est pas un échec : il dit
que les portes passent mais que la population est spatialement plus concentrée que le
cadrage — l'axe A9 du ticket 020, qui se corrige dans le **tirage**, pas ici.

Deux principes, hérités des tickets 015 et 019 :

- **aucun repli silencieux** : un script dont la ressource d'accès restreint est absente
  ou périmée **refuse de tourner** au lieu d'imputer à l'aveugle. Les ressources se
  (re)produisent avec `make zones`, `make housing-type`, `make bike-ownership` ;
- **un échec ici n'est pas un avertissement** : la population est alors **incomplète**, et
  l'audit de l'étape 9 le dit sans le nuancer.


In [15]:
# ── Étape 8 — Traits imputés depuis EMC² ─────────────────────────────────────
import subprocess

print('=' * 60)
print('ÉTAPE 8 — Traits imputés depuis les microdonnées EMC²')
print('=' * 60)

# Interpréteur des post-traitements. Ils importent `llm_module` (shapely, pyproj,
# geopandas) et `scripts.*` : le noyau du notebook n'a pas forcément ces dépendances,
# on privilégie donc le venv du projet, comme le fait le Makefile (SYNTHESIS_PYTHON).
_venv = REPO_ROOT / 'llm-agents' / '.venv' / 'bin' / 'python'
POST_PROCESS_PYTHON = str(_venv) if _venv.exists() else sys.executable
print(f'Interpréteur : {POST_PROCESS_PYTHON}')

# Ordre imposé, et deux contraintes plutôt qu'une :
#
# 1. `enrich_personal_bike` lit `housing_type`. `enrich_residence_zone` ne dépend de
#    rien et passe en tête des enrichissements.
# 2. `car_availability` DÉRIVE du nombre de permis du ménage (règle 4 de
#    `fix_minor_traits`). `enrich_equipment` réécrit les permis : tout
#    `car_availability` calculé AVANT lui est périmé, et rien ne le signalerait.
#    D'où la seconde passe de `fix_minor_traits` — il est idempotent, la rejouer ne
#    coûte que le recalcul — puis une vérification en lecture seule. C'est
#    l'avertissement explicite du ticket 017 ; l'inverser est le piège qu'il décrit.
POST_STEPS = [
    ('fix_minor_traits',     'scripts.data.population.fix_minor_traits',     []),
    ('enrich_residence_zone', 'scripts.data.population.enrich_residence_zone', ['--check']),
    ('enrich_housing_type',  'scripts.data.population.enrich_housing_type',  ['--check']),
    ('enrich_personal_bike', 'scripts.data.population.enrich_personal_bike', ['--check']),
    # Tickets 016 et 017 : abonnement TC et permis, appris sur EMC² (`make
    # equipment-propensity`). Pose seule ici — la recette vient après la 2e passe.
    ('enrich_equipment',     'scripts.data.population.enrich_equipment',     []),
    # 2e passe : recalcule `car_availability` sur les permis qui viennent d'être posés.
    ('fix_minor_traits (2e passe)', 'scripts.data.population.fix_minor_traits', []),
    # Recette en LECTURE SEULE, une fois `car_availability` remis d'accord. Le code 4
    # qu'elle peut rendre n'est pas un échec du trait : cf. la note ci-dessous.
    ('enrich_equipment (recette)', 'scripts.data.population.enrich_equipment',
     ['--dry-run', '--check']),
]

targets = [POP_DIR / pop_filename(n) for n in POPULATION_SIZES]
targets = [p for p in targets if p.exists()]
if not targets:
    raise FileNotFoundError(
        f'Aucune population dans {POP_DIR} — jouez l\'étape 7 (export final) d\'abord.')

_env = {**os.environ, 'PYTHONPATH': str(REPO_ROOT)}
POST_RESULTS = {}
for label, module, extra in POST_STEPS:
    print(f'\n── {label} ' + '─' * (56 - len(label)))
    proc = subprocess.run(
        [POST_PROCESS_PYTHON, '-m', module, *[str(p) for p in targets], *extra],
        cwd=str(REPO_ROOT), env=_env, capture_output=True, text=True,
    )
    print(proc.stdout.rstrip() or '(pas de sortie)')
    if proc.returncode != 0:
        # On n'interrompt PAS la boucle : chaque trait est indépendant, et savoir
        # lesquels manquent vaut mieux que s'arrêter au premier.
        #
        # Le code 3 n'est PAS un échec : il dit « enrichi, mais pas assez de foyers
        # pour trancher » — le cas normal des populations de 10 ou 100 agents, où un
        # croisement ne peut rien arbitrer. Le confondre avec un vrai échec apprend à
        # ignorer les échecs.
        if proc.returncode == 3:
            print('[NON VALIDÉ] population trop petite pour trancher — pas un échec')
        elif proc.returncode == 4:
            # Le code 4 n'est pas un échec non plus : les portes du trait passent, mais
            # la COMPOSITION de la population s'écarte de l'enquête. Deux cas le
            # rendent : la répartition spatiale (axe A9 du ticket 020) et, depuis
            # `enrich_equipment`, une strate d'occupation dont les covariables ne sont
            # pas celles de l'enquête. Les deux se corrigent dans le TIRAGE, pas dans
            # l'enrichissement.
            print('[ÉCART CADRAGE] portes passées, composition de la population '
                  'écartée du cadrage — se corrige au tirage, pas un défaut du trait')
        else:
            print(f'[ÉCHEC] code {proc.returncode}')
            tail = (proc.stderr or '').strip().split('\n')[-6:]
            for line in tail:
                print(f'        {line}')
    POST_RESULTS[label] = proc.returncode

print()
_verdict = {0: 'ok', 3: 'non validé (trop petit)', 4: 'ok (écart de cadrage A9)'}
print('Étape 8 terminée — ' + ' | '.join(
    f'{k}: {_verdict.get(v, f"ÉCHEC({v})")}' for k, v in POST_RESULTS.items()))

# Ce qui doit interrompre la chaîne : un trait qu'on n'a pas pu poser du tout.
_blocking = {k: v for k, v in POST_RESULTS.items() if v not in (0, 3, 4)}
if _blocking:
    print(f'\n[BLOQUANT] {", ".join(_blocking)} — la population est INCOMPLÈTE.')
    print('           Lisez la cause ci-dessus : elle dit quelle ressource produire')
    print('           (make zones / make housing-type / make bike-ownership /')
    print('            make communes-couronnes).')


ÉTAPE 8 — Traits imputés depuis les microdonnées EMC²
Interpréteur : /Users/yvesb/Documents/Projects/llm-agents-gama/llm-agents/.venv/bin/python

── fix_minor_traits ────────────────────────────────────────

=== /Users/yvesb/Documents/Projects/llm-agents-gama/data/population/toulouse_population_1000.json

  1021 personnes, 547 ménages (regroupés par coordonnées du domicile)
  permis retirés (< 18 ans)                131
  activités work → education               168
  travel_purposes recalculés               124
  car_availability recalculés              242
  VAE → vélo normal (< 14 ans)               9
  ⚠ number_of_cars divergent dans un ménage     6
  ⚠ ménages partiellement exportés         121
  ⚠ ménages : voiture, aucun conducteur     16

  État après correction :
    mineurs avec permis                      0  (cible 0)
    activités « education »                218  (cible > 120)
    activités « work »                     597
    car_availability : all 648, some 243, none 130

---
## Étape 9 — Audit : la population est-elle complète ?

Dernière cellule, et la seule qui prononce un verdict. Elle relit les fichiers
réellement déposés dans `data/population/` et vérifie la présence des traits que la
simulation consomme — pas leur justesse statistique (c'est le travail des `--check` de
l'étape 8), mais leur **présence**.

Le verdict est binaire et volontairement sévère : un trait manquant sur une part
significative des agents rend la population **inutilisable**, parce que les consommateurs
en aval ne s'en plaignent pas tous. `personal_bike` absent était traité comme « vélo
normal » jusqu'au ticket 015 — 100 % des agents à vélo, sans une ligne de log.


In [16]:
# ── Étape 9 — Audit de complétude ────────────────────────────────────────────
print('=' * 60)
print('ÉTAPE 9 — Audit de complétude')
print('=' * 60)

# Traits attendus dans traits_json, et la couverture minimale acceptable.
# `housing_type` et `personal_bike` sont légitimement absents hors couche de zones fines
# EMC² (~5 % des domiciles, communes franchement extérieures) : le seuil en tient compte.
EXPECTED_TRAITS = {
    'age':                 1.00,
    'gender':              1.00,
    'household_size':      1.00,
    'main_occupation':     1.00,
    'number_of_cars':      1.00,
    'has_driving_license': 1.00,
    'has_pt_subscription': 1.00,
    'housing_type':        0.90,
    'personal_bike':       0.90,
}

verdicts = {}
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    path  = POP_DIR / fname
    if not path.exists():
        print(f'\n[ABSENT] {fname} — non exporté')
        verdicts[fname] = False
        continue

    data = load_json(path)
    n = len(data)
    n_scheduled = sum(1 for p in data for a in p['identity']['activities']
                      if a.get('scheduled_start_time') is not None)
    print(f'\n{fname} — {n} agents, {n_scheduled} activités planifiées')

    ok = n_scheduled > 0
    if not ok:
        print('  [ÉCHEC] aucune activité planifiée : étape 5 ou 7 non jouée')
    for trait, threshold in EXPECTED_TRAITS.items():
        present = sum(1 for p in data
                      if (p['identity'].get('traits_json') or {}).get(trait) is not None)
        share = present / n if n else 0.0
        good = share >= threshold
        ok &= good
        flag = 'ok   ' if good else 'ÉCHEC'
        print(f'  {flag} {trait:22s} {present:6d}/{n} ({100 * share:5.1f} % '
              f'— seuil {100 * threshold:.0f} %)')
    verdicts[fname] = bool(ok)

print()
print('=' * 60)
for fname, ok in verdicts.items():
    print(f'{"POPULATION COMPLÈTE  " if ok else "POPULATION INCOMPLÈTE"}  {fname}')
print('=' * 60)
if not all(verdicts.values()):
    print('\nUn trait manquant n\'est pas un détail : les consommateurs en aval ne s\'en')
    print('plaignent pas tous. Reprenez l\'étape 8 et lisez la cause exacte — elle dit')
    print('quelle ressource d\'accès restreint produire (make zones / housing-type /')
    print('bike-ownership).')


ÉTAPE 9 — Audit de complétude

toulouse_population_1000.json — 1021 agents, 3944 activités planifiées
  ok    age                      1021/1021 (100.0 % — seuil 100 %)
  ok    gender                   1021/1021 (100.0 % — seuil 100 %)
  ok    household_size           1021/1021 (100.0 % — seuil 100 %)
  ok    main_occupation          1021/1021 (100.0 % — seuil 100 %)
  ok    number_of_cars           1021/1021 (100.0 % — seuil 100 %)
  ok    has_driving_license      1021/1021 (100.0 % — seuil 100 %)
  ok    has_pt_subscription      1021/1021 (100.0 % — seuil 100 %)
  ok    housing_type              976/1021 ( 95.6 % — seuil 90 %)
  ok    personal_bike             976/1021 ( 95.6 % — seuil 90 %)

POPULATION COMPLÈTE    toulouse_population_1000.json
